# 01 — Data Collection

End-to-end pull from Steam Storefront, Steam Web API, SteamSpy, IsThereAnyDeal, and SteamCharts into `data/steam.db`.

**Resumable.** Each stage updates progress flags on the `app_list` table, so re-running a stage only fetches the appids that haven't been done yet. Safe to interrupt and restart.

**Order matters:** sample pool → Steam details (filters out non-games) → reviews → SteamSpy → ITAD mappings → ITAD price history → SteamCharts historical CCU. SteamCharts must run *after* details, since `steamcharts_history` has a FK to `games`.

**Time budget:** with 5,000 games and conservative throttles, expect Storefront ~2–3h, ITAD ~1h, SteamCharts ~3h. Run overnight or in chunks via the `LIMIT` in each stage.

## Setup

In [2]:
import sys
import pandas as pd
from pathlib import Path

# Make `src` importable when launched from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import db, steam_api, steamspy_api, itad_api, steamcharts
from src.utils import load_keys

keys = load_keys()
print(f".env path: {keys.get('env_path') or 'not found'}")
print(f"ITAD key loaded: {bool(keys.get('itad'))}")

# Connect + ensure schema (creates data/steam.db if missing)
conn = db.connect()
db.init_db(conn)
tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
print(f"Tables ({len(tables)}): {tables}")

c:\Users\Sam\anaconda3\envs\steamsale\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


.env path: C:\Users\Sam\Documents\College\2026 Spring Term\DMW\Y2T2-Final-Project\.env
ITAD key loaded: True
Tables (15): ['app_list', 'cleaned_discount_panel', 'cleaned_games', 'cleaned_sale_events', 'game_categories', 'game_genres', 'games', 'itad_mapping', 'player_counts', 'price_history', 'review_timestamps', 'reviews_summary', 'steamcharts_history', 'steamspy', 'steamspy_tags']


## Stage 1 — Build the sample pool

Pull top-owned games from SteamSpy (5 pages × 1000 = ~5,000 candidates) and seed them into `app_list`. SteamSpy's `request=all` is rate-limited to 1 call per 60s, so this stage takes ~5 min.

In [2]:
pool = steamspy_api.fetch_sample_pool(num_pages=5)
print(f'pulled {len(pool)} candidate games from SteamSpy')

inserted = db.add_to_app_list(
    conn,
    [(p['appid'], p['name']) for p in pool],
    source='steamspy_top',
)
print(f'{inserted} new appids added to app_list')
print('app_list size:', conn.execute('SELECT COUNT(*) FROM app_list').fetchone()[0])

pulled 5000 candidate games from SteamSpy
0 new appids added to app_list
app_list size: 4995


## Stage 2 — Steam Storefront details

The bottleneck stage. Fetches `/api/appdetails` for each appid (cc=ph for PHP pricing) and decomposes the response into `games`, `game_genres`, `game_categories`. Non-`game` types (DLC, demo, soundtrack) get their flag set to 1 with `last_error='type=...'` so we don't retry them.

Run this cell repeatedly with a `LIMIT` if you want to checkpoint progress.

In [3]:
# Take a small batch first to verify the pipeline end-to-end before going big.
batch = db.pending_appids(conn, 'has_details', limit=20)
print(f'fetching details for {len(batch)} games...')
stats = steam_api.collect_app_details(conn, batch, cc='ph')
print(stats)

fetching details for 20 games...


Storefront details:   0%|          | 0/20 [00:00<?, ?game/s]

Storefront details: 100%|██████████| 20/20 [00:28<00:00,  1.44s/game]

{'ok': 0, 'missing': 20, 'skipped': 0, 'error': 0}


In [4]:
# Full run — uncomment when ready. ~1.5s per request, so 5000 games ≈ 2h.
batch = db.pending_appids(conn, 'has_details')
stats = steam_api.collect_app_details(conn, batch, cc='ph')
print(stats)

Storefront details: 100%|██████████| 46/46 [01:09<00:00,  1.50s/game]

{'ok': 0, 'missing': 46, 'skipped': 0, 'error': 0}


In [5]:
# Sanity check — what landed in `games`?
pd.read_sql('SELECT appid, title, type, currency, launch_price_cents, current_price_cents, discount_percent, release_date FROM games LIMIT 10', conn)

,appid,title,type,currency,launch_price_cents,current_price_cents,discount_percent,release_date
0,10,Counter-Strike,game,PHP,28995,28995,0,"1 Nov, 2000"
1,20,Team Fortress Classic,game,PHP,17495,17495,0,"1 Apr, 1999"
2,30,Day of Defeat,game,PHP,17495,17495,0,"1 May, 2003"
3,40,Deathmatch Classic,game,PHP,17495,17495,0,"1 Jun, 2001"
4,50,Half-Life: Opposing Force,game,PHP,17495,17495,0,"1 Nov, 1999"
5,60,Ricochet,game,PHP,17495,17495,0,"1 Nov, 2000"
6,70,Half-Life,game,PHP,28995,28995,0,"19 Nov, 1998"
7,80,Counter-Strike: Condition Zero,game,PHP,28995,28995,0,"1 Mar, 2004"
8,100,Counter-Strike: Condition Zero,game,PHP,28995,28995,0,"1 Mar, 2004"
9,130,Half-Life: Blue Shift,game,PHP,17495,17495,0,"1 Jun, 2001"


## Stage 3 — Review summaries

Aggregate review counts + score per game. Set `fetch_individual=True` to also pull paginated review timestamps (used for review-velocity in Part 2). Pulling individual reviews multiplies the request count, so leave it off for the first pass.

In [6]:
batch = db.pending_appids_with_game(conn, 'has_reviews', limit=50)
stats = steam_api.collect_reviews(conn, batch, fetch_individual=False)
print(stats)

Reviews: 0game [00:00, ?game/s]

{'ok': 0, 'missing': 0, 'error': 0}


In [7]:
# When you're ready for review timestamps (Part 2):
# batch = db.pending_appids_with_game(conn, 'has_reviews')
# steam_api.collect_reviews(conn, batch, fetch_individual=True, max_pages=5)

## Stage 4 — SteamSpy per-app details

Owners estimates, playtime medians, and community tags. Throttled at ~1/sec — 5000 games ≈ 90 min.

In [8]:
batch = db.pending_appids_with_game(conn, 'has_steamspy', limit=50)
stats = steamspy_api.collect_steamspy(conn, batch)
print(stats)

SteamSpy: 0game [00:00, ?game/s]

{'ok': 0, 'missing': 0, 'error': 0}


## Stage 5 — IsThereAnyDeal

Two sub-stages: first translate Steam appid → ITAD UUID, then pull historical price points (Steam shop only — `shop_id=61`) for the games we successfully mapped.

**Country choice — `country='US'` for deepest history.** ITAD's `/games/history/v2` endpoint caps the response at the last 3 months by default (the `since` parameter overrides this). On top of that, regional coverage varies dramatically: PH returns ~3 events per game total, US returns 20-100+ events going back to 2012-2015 for typical Steam games. We default to US because the `cut` (discount percent) field is currency-agnostic — a 50% discount is 50% in any currency. Absolute-price columns (`price_amount`, `regular_amount`) end up in USD, so downstream analyses should rely on `cut` rather than mixing currencies. (Steam Storefront snapshots in the `games` table are still queried with `cc=ph`, so the launch/current-price columns there are PHP centavos as before.)

**Re-running for fresh data.** Once a game has `has_price_history = 1`, the collector skips it on subsequent runs. To force a full refresh — e.g., after switching country or `since` — set `RESET_FIRST = True` in the price-history cell below. That clears the `price_history` table and resets the flags so every game is re-fetched. ITAD coverage may still miss obscure indies regardless of country.

In [9]:
if keys.get('itad'):
    batch = db.pending_appids_with_game(conn, 'has_itad_id', limit=50)
    stats = itad_api.collect_itad_mappings(conn, keys['itad'], batch)
    print(stats)
else:
    print('ITAD key missing; skipping mapping collection')

ITAD lookup: 100%|██████████| 1/1 [00:01<00:00,  1.02s/game]

{'ok': 0, 'missing': 1, 'error': 0}


In [4]:
# Optional: clear the existing price_history and re-fetch from scratch.
# Set to True if you've changed country/since defaults and want fresh data
# for every game (otherwise the collector skips games already marked done).
RESET_FIRST = False

if keys.get('itad'):
    if RESET_FIRST:
        itad_api.reset_price_history(conn)

    # country='US' + since=2010 (defaults in itad_api) -> 20-100+ events per
    # game going back to 2012-2015, vs ~3 events for PH. The `cut` field is
    # currency-agnostic so percentage-based analysis is unaffected.
    stats = itad_api.collect_price_history(conn, keys['itad'], shops=[61])
    print(stats)
else:
    print('ITAD key missing; skipping price history collection')

ITAD price history: 100%|██████████| 4661/4661 [1:20:29<00:00,  1.04s/game]

{'ok': 4620, 'empty': 41, 'error': 0, 'rows_inserted': 409977}


## Stage 6 — SteamCharts historical CCU

Scrapes `steamcharts.com/app/{appid}/chart-data.json` for ~weekly historical concurrent-player counts going back to 2012. This is the primary input for the Part 2 sale-uplift analysis: for each ITAD sale event, we'll measure CCU in the sale window vs. a pre-sale baseline.

Throttled to 2s/req with a descriptive User-Agent. ~3h for 5,000 games. Uses `pending_appids_with_game` so only appids with a `games` row are attempted (FK requirement).

In [11]:
# Small batch first — SteamCharts is throttled at 2s/req.
batch = db.pending_appids_with_game(conn, 'has_steamcharts', limit=20)
print(f'fetching CCU history for {len(batch)} games...')
stats = steamcharts.collect_history(conn, batch)
print(stats)

fetching CCU history for 20 games...


SteamCharts:   5%|▌         | 1/20 [00:00<00:07,  2.57game/s]

[get_with_retry] 404 on https://steamcharts.com/app/3483/chart-data.json: <!DOCTYPE html>
<html lang="en">
<head>

<meta charset="utf-8">
<title>Steam Charts - Tracking What's Played</title>
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<meta name="


SteamCharts:  10%|█         | 2/20 [00:02<00:24,  1.34s/game]

[get_with_retry] 404 on https://steamcharts.com/app/33229/chart-data.json: <!DOCTYPE html>
<html lang="en">
<head>

<meta charset="utf-8">
<title>Steam Charts - Tracking What's Played</title>
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<meta name="


...

[get_with_retry] 404 on https://steamcharts.com/app/901399/chart-data.json: <!DOCTYPE html>
<html lang="en">
<head>

<meta charset="utf-8">
<title>Steam Charts - Tracking What's Played</title>
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<meta name="


SteamCharts:  95%|█████████▌| 19/20 [00:36<00:01,  2.00s/game]

[get_with_retry] 404 on https://steamcharts.com/app/901583/chart-data.json: <!DOCTYPE html>
<html lang="en">
<head>

<meta charset="utf-8">
<title>Steam Charts - Tracking What's Played</title>
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<meta name="


SteamCharts: 100%|██████████| 20/20 [00:38<00:00,  1.92s/game]

[get_with_retry] 404 on https://steamcharts.com/app/1098292/chart-data.json: <!DOCTYPE html>
<html lang="en">
<head>

<meta charset="utf-8">
<title>Steam Charts - Tracking What's Played</title>
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<meta name="
{'ok': 0, 'missing': 20, 'error': 0, 'rows_inserted': 0}


## Sanity checks

In [12]:
summary = pd.read_sql('''
    SELECT
      (SELECT COUNT(*) FROM app_list)              AS app_list,
      (SELECT COUNT(*) FROM games)                 AS games,
      (SELECT COUNT(*) FROM game_genres)           AS genre_rows,
      (SELECT COUNT(*) FROM game_categories)       AS category_rows,
      (SELECT COUNT(*) FROM reviews_summary)       AS reviews_summary,
      (SELECT COUNT(*) FROM review_timestamps)     AS review_rows,
      (SELECT COUNT(*) FROM steamspy)              AS steamspy,
      (SELECT COUNT(*) FROM steamspy_tags)         AS steamspy_tags,
      (SELECT COUNT(*) FROM itad_mapping)          AS itad_mapping,
      (SELECT COUNT(*) FROM price_history)         AS price_history_rows,
      (SELECT COUNT(*) FROM steamcharts_history)   AS steamcharts_rows
''', conn)
summary.T.rename(columns={0: 'rows'})

,rows
app_list,4995
games,4946
genre_rows,13753
category_rows,37358
reviews_summary,4946
review_rows,477328
steamspy,4946
steamspy_tags,89072
itad_mapping,4945
price_history_rows,19006


In [13]:
# Per-stage progress on the worklist
pd.read_sql('''
    SELECT
      SUM(has_details)        AS details_done,
      SUM(has_reviews)        AS reviews_done,
      SUM(has_steamspy)       AS steamspy_done,
      SUM(has_itad_id)        AS itad_id_done,
      SUM(has_price_history)  AS price_history_done,
      SUM(has_steamcharts)    AS steamcharts_done,
      COUNT(*)                AS total
    FROM app_list
''', conn)

,details_done,reviews_done,steamspy_done,itad_id_done,price_history_done,steamcharts_done,total
0,4949,4946,4946,4945,4945,4923,4995


In [ ]:
conn.close()
print('connection closed')